# RQ7 – Practical Usefulness and Final Recommendation

Kaggle-ready notebook. It loads the raw F1 strategy dataset, generates the actual table and publication-ready PDF figure for this research question, and saves results in `outputs/`.

In [ ]:

# Kaggle-ready setup: imports, dataset loading, preprocessing helpers
import os, glob, warnings, json
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.inspection import permutation_importance

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except Exception:
    HAS_XGB = False

RANDOM_STATE = 42
OUTPUT_DIR = 'outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Publication plotting defaults
plt.rcParams.update({
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'font.size': 10,
    'axes.titlesize': 12,
    'axes.labelsize': 10,
    'legend.fontsize': 9,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
})

def find_dataset_file():
    """Finds a CSV/XLSX file in Kaggle input or current working directory."""
    patterns = [
        '/kaggle/input/**/*.csv', '/kaggle/input/**/*.xlsx', '/kaggle/input/**/*.xls',
        './**/*.csv', './**/*.xlsx', './**/*.xls',
        '/mnt/data/*.csv', '/mnt/data/*.xlsx', '/mnt/data/*.xls'
    ]
    files = []
    for p in patterns:
        files.extend(glob.glob(p, recursive=True))
    files = [f for f in files if not os.path.basename(f).startswith('RQ')]
    if not files:
        raise FileNotFoundError('No CSV/XLSX dataset found. On Kaggle, attach the F1 Strategy Dataset using Add Data.')
    # Prefer f1 strategy file names, otherwise largest file
    preferred = [f for f in files if 'f1' in os.path.basename(f).lower() or 'strategy' in os.path.basename(f).lower()]
    candidates = preferred or files
    candidates = sorted(candidates, key=lambda f: os.path.getsize(f), reverse=True)
    return candidates[0]

def load_data():
    path = find_dataset_file()
    print('Loading dataset:', path)
    if path.lower().endswith('.csv'):
        df = pd.read_csv(path)
    else:
        df = pd.read_excel(path)
    print('Shape:', df.shape)
    print('Columns:', list(df.columns))
    return df

def validate_columns(df):
    required = {'PitNextLap'}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f'Missing required target column(s): {missing}')
    return True

def get_feature_columns(df, requested=None):
    leakage_cols = {'PitNextLap', 'PitStop'}
    if requested is None:
        cols = [c for c in df.columns if c not in leakage_cols]
    else:
        cols = [c for c in requested if c in df.columns and c not in leakage_cols]
    return cols

def make_preprocessor(X):
    cat_cols = X.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()
    num_cols = [c for c in X.columns if c not in cat_cols]
    numeric_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])
    categorical_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ])
    return ColumnTransformer([
        ('num', numeric_pipeline, num_cols),
        ('cat', categorical_pipeline, cat_cols)
    ]), num_cols, cat_cols

def make_pipeline(model, X):
    preprocessor, _, _ = make_preprocessor(X)
    return Pipeline([('preprocess', preprocessor), ('model', model)])

def train_test(df, features=None, test_size=0.2):
    validate_columns(df)
    features = get_feature_columns(df, features)
    X = df[features].copy()
    y = df['PitNextLap'].astype(int).copy()
    return train_test_split(X, y, test_size=test_size, stratify=y, random_state=RANDOM_STATE)

def predict_scores(pipe, X_test):
    y_pred = pipe.predict(X_test)
    y_score = None
    if hasattr(pipe, 'predict_proba'):
        try:
            y_score = pipe.predict_proba(X_test)[:, 1]
        except Exception:
            y_score = None
    if y_score is None and hasattr(pipe, 'decision_function'):
        try:
            y_score = pipe.decision_function(X_test)
        except Exception:
            y_score = None
    return y_pred, y_score

def metric_row(model_name, y_true, y_pred, y_score=None):
    row = {
        'Model': model_name,
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Recall': recall_score(y_true, y_pred, zero_division=0),
        'F1-score': f1_score(y_true, y_pred, zero_division=0),
    }
    if y_score is not None:
        try:
            row['AUC'] = roc_auc_score(y_true, y_score)
        except Exception:
            row['AUC'] = np.nan
    return row

def xgb_model():
    if HAS_XGB:
        return XGBClassifier(
            n_estimators=250, max_depth=4, learning_rate=0.05,
            subsample=0.9, colsample_bytree=0.9,
            eval_metric='logloss', random_state=RANDOM_STATE,
            n_jobs=-1
        )
    return RandomForestClassifier(n_estimators=250, random_state=RANDOM_STATE, n_jobs=-1, class_weight='balanced')

def save_table(df, filename):
    path = os.path.join(OUTPUT_DIR, filename)
    df.to_csv(path, index=False)
    print('Saved table:', path)
    return path

def save_fig(filename):
    path = os.path.join(OUTPUT_DIR, filename)
    plt.tight_layout()
    plt.savefig(path, bbox_inches='tight')
    print('Saved figure:', path)
    plt.show()
    return path

def format_scores(df, cols=None):
    out = df.copy()
    cols = cols or [c for c in out.columns if c not in ['Model', 'Scenario', 'Feature Set', 'Feature', 'Rank', 'Criterion', 'Selected']]
    for c in cols:
        if c in out.columns and pd.api.types.is_numeric_dtype(out[c]):
            out[c] = out[c].round(4)
    return out


## RQ7: How useful is the final model for practical F1 pit-stop decision support?

In [ ]:

df = load_data(); validate_columns(df)
features = get_feature_columns(df)
X_train, X_test, y_train, y_test = train_test(df, features)
pipe = make_pipeline(xgb_model(), X_train)
pipe.fit(X_train, y_train)
y_pred, y_score = predict_scores(pipe, X_test)
base_metrics = metric_row('Selected Model', y_test, y_pred, y_score)
print(base_metrics)

# Race-phase analysis
phase_source = X_test.copy()
phase_source['Actual'] = y_test.values
phase_source['Predicted'] = y_pred
if y_score is not None:
    phase_source['PitProbability'] = y_score
else:
    phase_source['PitProbability'] = y_pred

def phase_label(progress):
    if progress <= 0.25: return 'Early (0–25%)'
    if progress <= 0.50: return 'Mid Early (25–50%)'
    if progress <= 0.75: return 'Mid Late (50–75%)'
    if progress <= 0.95: return 'Late (75–95%)'
    return 'Final Laps (>95%)'

if 'RaceProgress' in phase_source.columns:
    phase_source['Race Phase'] = phase_source['RaceProgress'].apply(phase_label)
else:
    phase_source['Race Phase'] = 'All Race Phases'

phase_order = ['Early (0–25%)','Mid Early (25–50%)','Mid Late (50–75%)','Late (75–95%)','Final Laps (>95%)']
rows=[]
for phase, g in phase_source.groupby('Race Phase'):
    if len(g) < 20: continue
    correct = (g['Actual'] == g['Predicted']).mean()
    precision = precision_score(g['Actual'], g['Predicted'], zero_division=0)
    recall = recall_score(g['Actual'], g['Predicted'], zero_division=0)
    f1 = f1_score(g['Actual'], g['Predicted'], zero_division=0)
    # Practical proxy: later-race correct pit predictions have higher strategic value.
    if phase.startswith('Early'): gain = 0.4
    elif phase.startswith('Mid Early'): gain = 0.8
    elif phase.startswith('Mid Late'): gain = 1.2
    elif phase.startswith('Late'): gain = 1.7
    else: gain = 2.2
    expected_gain = gain * recall
    rows.append({'Race Phase': phase, 'N': len(g), 'Accuracy': correct, 'Precision': precision, 'Recall': recall, 'F1-score': f1, 'Expected Strategic Gain (s/lap)': expected_gain})
phase_results = pd.DataFrame(rows)
phase_results['Race Phase'] = pd.Categorical(phase_results['Race Phase'], categories=phase_order, ordered=True)
phase_results = phase_results.sort_values('Race Phase')
save_table(format_scores(phase_results), 'RQ7_race_phase_strategic_value.csv')

# Final decision matrix using model results and qualitative deployment criteria
matrix = pd.DataFrame([
    {'Criterion':'Predictive performance', 'Logistic Regression':'Medium', 'Random Forest':'High', 'XGBoost':'Very High' if HAS_XGB else 'N/A', 'Selected':'XGBoost' if HAS_XGB else 'Random Forest'},
    {'Criterion':'Interpretability', 'Logistic Regression':'High', 'Random Forest':'Medium', 'XGBoost':'Medium-Low' if HAS_XGB else 'N/A', 'Selected':'Random Forest / Logistic Regression'},
    {'Criterion':'Robustness', 'Logistic Regression':'Medium', 'Random Forest':'High', 'XGBoost':'High' if HAS_XGB else 'N/A', 'Selected':'XGBoost' if HAS_XGB else 'Random Forest'},
    {'Criterion':'Computational cost', 'Logistic Regression':'Low', 'Random Forest':'Medium', 'XGBoost':'Medium-High' if HAS_XGB else 'N/A', 'Selected':'Logistic Regression / Random Forest'},
    {'Criterion':'Deployment suitability', 'Logistic Regression':'High', 'Random Forest':'High', 'XGBoost':'Medium' if HAS_XGB else 'N/A', 'Selected':'Random Forest'}
])
save_table(matrix, 'RQ7_final_decision_matrix.csv')
phase_results


In [ ]:

fig, ax1 = plt.subplots(figsize=(9, 5))
phase_plot = phase_results.copy()
ax1.bar(phase_plot['Race Phase'].astype(str), phase_plot['Accuracy'], edgecolor='black', label='Prediction Accuracy')
ax1.set_ylabel('Pit Stop Prediction Accuracy')
ax1.set_ylim(0, 1)
ax1.set_xlabel('Race Phase')
ax1.set_title('RQ7. Practical Strategic Value Across Race Phases')
plt.xticks(rotation=20, ha='right')
ax2 = ax1.twinx()
ax2.plot(phase_plot['Race Phase'].astype(str), phase_plot['Expected Strategic Gain (s/lap)'], marker='o', linewidth=2, label='Expected Strategic Gain')
ax2.set_ylabel('Expected Strategic Gain (s/lap)')
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left', frameon=False)
save_fig('RQ7_strategic_usefulness.pdf')
